# Module 5.8 — Model Merging: SLERP & DARE-TIES

> **Optional module.** Prerequisites: both `deskmate-sft-merged/` and `deskmate-dpo-merged/` must exist (Modules 3.4 and 3.7 adapters merged into base).

**Goal:** Combine the SFT checkpoint (high ROUGE-L, accurate content) with the DPO checkpoint (better tone/format) into a single merged model — zero extra training, zero latency change.

## 1. Install Dependencies

In [ ]:
# !pip install -q mergekit transformers accelerate rouge-score

import os, json, copy, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path

SMOKE_TEST  = True   # set False for real merges
RUN_JUDGE   = False  # set True to run LLM-as-judge tone scorer

BASE_MODEL  = 'Qwen/Qwen2.5-1.5B-Instruct'
SFT_MODEL   = 'models/deskmate-sft-merged/'
DPO_MODEL   = 'models/deskmate-dpo-merged/'
LINEAR_OUT  = 'models/deskmate-linear-merged/'
SLERP_OUT   = 'models/deskmate-slerp-merged/'
TIES_OUT    = 'models/deskmate-dare-ties-merged/'

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)
print('Config OK')

## 2. HF Token

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
print('Token loaded:', bool(HF_TOKEN))

## 3. Gold Evaluation Set & Helpers

In [ ]:
from rouge_score import rouge_scorer as rs_module

_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

GOLD = [
    {'ticket': 'I was charged twice for my subscription last month.',
     'ref': 'Contact support with your invoice numbers. We will investigate and refund within 3 business days.'},
    {'ticket': 'How do I reset my 2FA device?',
     'ref': 'Go to Account > Security > Two-Factor Authentication and click Reset Device.'},
    {'ticket': 'The CSV export button is not working.',
     'ref': 'This is a known issue ERR-500 fixed in version 4.3.0. Please update or contact support.'},
    {'ticket': 'Can I get a refund after 30 days?',
     'ref': 'DeskMate offers a 30-day money-back guarantee. Refunds after 30 days are evaluated case by case.'},
    {'ticket': 'My iOS app crashes on iPhone 14.',
     'ref': 'A crash affecting iOS 17 was fixed in app version 2.1.3. Please update from the App Store.'},
]

def rouge_l(pred, ref):
    return round(_scorer.score(ref, pred)['rougeL'].fmeasure, 4)

def mean_rouge(replies):
    return round(sum(rouge_l(r, ex['ref']) for r, ex in zip(replies, GOLD)) / len(GOLD), 4)

print(f'Gold examples: {len(GOLD)}')

## 4. Task Vectors — What Merging Combines

A **task vector** is the weight delta introduced by fine-tuning:
```
τ_sft = θ_sft − θ_base
τ_dpo = θ_dpo − θ_base
```
Linear merge: `θ_merged = θ_base + λ·τ_sft + (1−λ)·τ_dpo`

The challenge: where SFT and DPO delta weights point in **opposite directions** for the same parameter, they cancel — degrading both capabilities. DARE-TIES resolves this by pruning low-signal deltas and resolving sign conflicts.

In [ ]:
def compute_task_vector(model_state, base_state):
    """Returns {name: delta_tensor} for each parameter."""
    return {k: model_state[k].float() - base_state[k].float()
            for k in base_state if k in model_state}

def task_vector_norm(tv):
    total = sum(t.pow(2).sum().item() for t in tv.values())
    return round(total ** 0.5, 2)

if SMOKE_TEST:
    print('SMOKE_TEST=True — skipping task vector computation.')
    print('[Simulated] ||τ_sft||: 14.32   ||τ_dpo||: 11.87')
    tv_sft_norm = 14.32
    tv_dpo_norm = 11.87
else:
    from transformers import AutoModelForCausalLM
    print('Loading base model state...')
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float32, token=HF_TOKEN)
    base_sd = {k: v.clone() for k, v in base.state_dict().items()}

    print('Loading SFT model state...')
    sft = AutoModelForCausalLM.from_pretrained(
        SFT_MODEL, torch_dtype=torch.float32)
    tv_sft = compute_task_vector(sft.state_dict(), base_sd)
    tv_sft_norm = task_vector_norm(tv_sft)

    print('Loading DPO model state...')
    dpo = AutoModelForCausalLM.from_pretrained(
        DPO_MODEL, torch_dtype=torch.float32)
    tv_dpo = compute_task_vector(dpo.state_dict(), base_sd)
    tv_dpo_norm = task_vector_norm(tv_dpo)

    del sft, dpo; torch.cuda.empty_cache()

print(f'||τ_sft||: {tv_sft_norm}   ||τ_dpo||: {tv_dpo_norm}')

## 5. Linear Merge (Baseline) — λ=0.5

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating linear merge.')
    linear_rouge = 0.455
    linear_tone  = 3.8
    print(f'[Simulated] Linear merge ROUGE-L: {linear_rouge}')
else:
    # Manual linear merge
    merged_sd = {}
    for k in base_sd:
        merged_sd[k] = (base_sd[k].float()
                        + 0.5 * tv_sft[k]
                        + 0.5 * tv_dpo[k]).half()

    base.load_state_dict(merged_sd)
    base.save_pretrained(LINEAR_OUT)
    print('Linear merge saved to', LINEAR_OUT)

## 6. SLERP Merge (mergekit)

In [ ]:
# SLERP interpolates along the geodesic on the unit sphere — preserving weight magnitude.
# Use mergekit CLI or Python API:

SLERP_CONFIG = {
    'merge_method': 'slerp',
    'models': [
        {'model': SFT_MODEL},
        {'model': DPO_MODEL},
    ],
    'base_model': BASE_MODEL,
    'parameters': {'t': 0.5},
    'dtype': 'float16',
    'out_path': SLERP_OUT,
}

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating SLERP merge.')
    print('SLERP config:', json.dumps(SLERP_CONFIG, indent=2))
    slerp_rouge = 0.463
    slerp_tone  = 4.1
    print(f'[Simulated] SLERP ROUGE-L: {slerp_rouge}  tone: {slerp_tone}')
else:
    # Run mergekit
    import yaml
    cfg_path = '/tmp/slerp_config.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(SLERP_CONFIG, f)
    os.system(f'mergekit-yaml {cfg_path} {SLERP_OUT} --cuda')
    print('SLERP merge complete:', SLERP_OUT)

## 7. DARE-TIES Merge (mergekit)

In [ ]:
# DARE: drop 90% lowest-magnitude delta weights, rescale remainder.
# TIES: elect sign by majority vote; average only sign-agreeing deltas.

TIES_CONFIG = {
    'merge_method': 'dare_ties',
    'models': [
        {'model': SFT_MODEL, 'parameters': {'weight': 0.5, 'density': 0.1}},
        {'model': DPO_MODEL, 'parameters': {'weight': 0.5, 'density': 0.1}},
    ],
    'base_model': BASE_MODEL,
    'dtype': 'float16',
    'out_path': TIES_OUT,
}

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating DARE-TIES merge.')
    print('DARE-TIES config:', json.dumps(TIES_CONFIG, indent=2))
    ties_rouge = 0.471
    ties_tone  = 4.2
    print(f'[Simulated] DARE-TIES ROUGE-L: {ties_rouge}  tone: {ties_tone}')
else:
    cfg_path = '/tmp/ties_config.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(TIES_CONFIG, f)
    os.system(f'mergekit-yaml {cfg_path} {TIES_OUT} --cuda')
    print('DARE-TIES merge complete:', TIES_OUT)

## 8. SLERP t Sweep — Find Optimal Blend

In [ ]:
T_VALS = [0.2, 0.4, 0.5, 0.6, 0.8]

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating t sweep.')
    # Quality peaks near t=0.4 (slightly SFT-dominant for ROUGE-L)
    t_rouge = {0.2: 0.471, 0.4: 0.468, 0.5: 0.463, 0.6: 0.455, 0.8: 0.441}
    t_tone  = {0.2: 3.5,   0.4: 3.9,   0.5: 4.1,   0.6: 4.3,   0.8: 4.5}
    print('t     | ROUGE-L | Tone')
    for t in T_VALS:
        print(f'{t}   | {t_rouge[t]}  | {t_tone[t]}')
else:
    t_rouge = {}
    t_tone  = {}
    # Full sweep omitted for brevity — run SLERP at each t value and evaluate
    print('Full t sweep requires repeated mergekit runs — see theory .md for methodology.')

best_t = max(t_rouge, key=lambda t: t_rouge[t] + 0.1 * t_tone[t])
print(f'\nBest t={best_t}: ROUGE-L={t_rouge[best_t]}, Tone={t_tone[best_t]}')

## 9. Summary: All Merge Methods vs Parents

In [ ]:
import pandas as pd

# Simulated parent benchmarks
sft_rouge  = 0.473;  sft_tone  = 3.6
dpo_rouge  = 0.461;  dpo_tone  = 4.4

df = pd.DataFrame({
    'Model':   ['SFT (parent)', 'DPO (parent)', 'Linear λ=0.5', 'SLERP t=0.5', 'DARE-TIES d=0.1'],
    'ROUGE-L': [sft_rouge, dpo_rouge, linear_rouge, slerp_rouge, ties_rouge],
    'Tone (1-5)': [sft_tone, dpo_tone, linear_tone, slerp_tone, ties_tone],
    'Latency': ['baseline', 'same', 'same', 'same', 'same'],
})
print(df.to_string(index=False))

## 10. Interference Check

In [ ]:
min_parent_rouge = min(sft_rouge, dpo_rouge)
ROUGE_GATE = min_parent_rouge - 0.02

print('Interference check (merged ROUGE-L must be >= min(parents) - 0.02):')
print(f'  min(parent ROUGE-L): {min_parent_rouge}')
print(f'  Gate threshold:      {ROUGE_GATE}')
print()
for name, r in [('Linear', linear_rouge), ('SLERP', slerp_rouge), ('DARE-TIES', ties_rouge)]:
    status = 'PASS' if r >= ROUGE_GATE else 'FAIL — WEIGHT INTERFERENCE DETECTED'
    print(f'  {name}: {r}  [{status}]')
print()
print('If any model FAILS: increase DARE density sparsity (density: 0.05)')
print('or adjust SLERP t toward the stronger parent.')

## 11. Quality Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

models = ['SFT', 'DPO', 'Linear', 'SLERP', 'DARE-TIES']
rouge_vals = [sft_rouge, dpo_rouge, linear_rouge, slerp_rouge, ties_rouge]
tone_vals  = [sft_tone,  dpo_tone,  linear_tone,  slerp_tone,  ties_tone]
colors = ['steelblue', 'coral', 'grey', 'seagreen', 'darkorange']

ax = axes[0]
bars = ax.bar(models, rouge_vals, color=colors)
ax.axhline(min_parent_rouge - 0.02, color='red', linestyle='--', linewidth=1,
           label='Interference gate')
ax.set_ylabel('ROUGE-L')
ax.set_title('Domain Accuracy (ROUGE-L)')
ax.set_ylim(0.42, 0.50)
ax.legend(fontsize=8)
for bar, v in zip(bars, rouge_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            str(v), ha='center', fontsize=8)

ax = axes[1]
bars = ax.bar(models, tone_vals, color=colors)
ax.set_ylabel('Tone rating (1-5)')
ax.set_title('Tone Quality (LLM judge)')
ax.set_ylim(2.5, 5.0)
for bar, v in zip(bars, tone_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(v), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('merge_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: merge_comparison.png')

## 12. Checkpoint

In [ ]:
checkpoint = '''
Checkpoint: What symptom tells you that weight interference is degrading the
merged model, and what is your next step?

Symptom:
  ROUGE-L on the gold set is lower than BOTH parent models -- the merged model
  is worse at domain accuracy than the SFT checkpoint AND worse at tone than the
  DPO checkpoint. The regression suite pass rate also drops below either parent
  alone. This means the SFT and DPO task vectors are cancelling in the directions
  that matter most for DeskMate.

Next step:
  Increase DARE sparsity: set density from 0.1 to 0.05, keeping only the top 5%
  highest-magnitude delta weights. This removes more of the low-signal overlap
  between the two task vectors, reducing interference. Re-benchmark after each
  change. If interference persists, adjust the TIES weight ratio toward the
  stronger parent (e.g. weight: 0.7 for SFT, 0.3 for DPO).
'''
print(checkpoint)

## 13. Save Report & Best Merged Model

In [ ]:
# Choose the best merge method
best_merge = max(
    [('Linear', linear_rouge, linear_tone),
     ('SLERP',  slerp_rouge,  slerp_tone),
     ('DARE-TIES', ties_rouge, ties_tone)],
    key=lambda x: x[1] + 0.1 * x[2]  # weighted: ROUGE-L dominates, tone secondary
)
print(f'Best merge method: {best_merge[0]}')
print(f'  ROUGE-L: {best_merge[1]}   Tone: {best_merge[2]}')

report = [
    '# Model Merging Report\n',
    f'Base: {BASE_MODEL}',
    f'SFT:  {SFT_MODEL}',
    f'DPO:  {DPO_MODEL}',
    f'Smoke test: {SMOKE_TEST}\n',
    '| Model | ROUGE-L | Tone (1-5) | Latency |',
    '|-------|---------|------------|---------|',
    f'| SFT parent      | {sft_rouge} | {sft_tone} | baseline |',
    f'| DPO parent      | {dpo_rouge} | {dpo_tone} | same |',
    f'| Linear λ=0.5    | {linear_rouge} | {linear_tone} | same |',
    f'| SLERP t=0.5     | {slerp_rouge} | {slerp_tone} | same |',
    f'| DARE-TIES d=0.1 | {ties_rouge} | {ties_tone} | same |',
    '',
    f'Best method: {best_merge[0]}',
    '',
    '## Checkpoint',
    '',
    'Symptom of interference: merged ROUGE-L < both parents.',
    'Fix: increase DARE density sparsity (0.1 -> 0.05) to reduce delta overlap.',
]

with open('reports/merge_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/merge_report.md')
print('\n=== Module 5.8 Complete ===')
print('Phase 5 — Optimization for Production: ALL MODULES DONE')